# 01 — Data Exploration

This notebook walks through the Bacteria-2033Images-33Types dataset:
- Class distribution
- Sample image grid
- Image statistics (size, channels, pixel intensity)
- Colour histogram analysis


In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

from bacteria_classifier.dataset import BacteriaDataset
from bacteria_classifier.utils import plot_class_distribution, visualize_samples

# ── Change this to your extracted dataset path ──
DATA_ROOT = Path('../data/bacteria')

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 120

print(f'Dataset path: {DATA_ROOT.resolve()}')
print(f'Exists: {DATA_ROOT.exists()}')

## 1. Class Distribution

In [ ]:
ds = BacteriaDataset(DATA_ROOT)
counts = ds.class_counts()

print(f'Total images : {len(ds)}')
print(f'Total classes: {len(ds.classes)}')
print(f'Mean per class: {np.mean(list(counts.values())):.1f}')
print(f'Min / Max    : {min(counts.values())} / {max(counts.values())}')

In [ ]:
fig = plot_class_distribution(DATA_ROOT)
plt.show()

## 2. Sample Image Grid

In [ ]:
# Show 4 samples from the first 8 classes
fig = visualize_samples(DATA_ROOT, classes=ds.classes[:8], n_per_class=4)
plt.show()

## 3. Image Size Distribution

In [ ]:
widths, heights = [], []
for img_path, _ in tqdm(ds.samples[:200], desc='Scanning sizes'):
    with Image.open(img_path) as img:
        widths.append(img.width)
        heights.append(img.height)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=30, color='seagreen', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')
plt.tight_layout()
plt.show()

print(f'Width  — mean: {np.mean(widths):.0f}px  std: {np.std(widths):.0f}px')
print(f'Height — mean: {np.mean(heights):.0f}px  std: {np.std(heights):.0f}px')

## 4. Pixel Intensity & Colour Histograms

In [ ]:
# Sample 50 images and compute per-channel statistics
import random
random.seed(42)
sample_paths = [ds.samples[i][0] for i in random.sample(range(len(ds)), 50)]

r_hist = np.zeros(256)
g_hist = np.zeros(256)
b_hist = np.zeros(256)

for p in tqdm(sample_paths, desc='Computing histograms'):
    img = np.array(Image.open(p).convert('RGB'))
    r_hist += np.bincount(img[:,:,0].ravel(), minlength=256)
    g_hist += np.bincount(img[:,:,1].ravel(), minlength=256)
    b_hist += np.bincount(img[:,:,2].ravel(), minlength=256)

x = np.arange(256)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x, r_hist / r_hist.sum(), color='red',   alpha=0.8, label='Red')
ax.plot(x, g_hist / g_hist.sum(), color='green', alpha=0.8, label='Green')
ax.plot(x, b_hist / b_hist.sum(), color='blue',  alpha=0.8, label='Blue')
ax.set_xlabel('Pixel Intensity')
ax.set_ylabel('Normalised Frequency')
ax.set_title('Aggregate Colour Histogram (50 random images)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()